<a href="https://colab.research.google.com/github/martindiarua/ML_01/blob/main/work/notebooks/w04_signal_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/martindiarua/ML_01/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

Five signals relevant to content refresh priority:

- `days_since_last_update` — how long it has been since the page was updated
- `ctr` — the percentage of impressions that resulted in clicks
- `avg_position` — the page's average search position
- `search_volume` — estimated search demand
- `engagement_rate` — the proportion of sessions showing engagement

These fields have different ranges and distributions. Some search and traffic measures may be heavily concentrated, with a smaller number of pages having much larger values than most pages. I will inspect their distributions before choosing thresholds for the baseline ranking rule.

In [ ]:
signals = [
    "days_since_last_update",
    "ctr",
    "avg_position",
    "search_volume",
    "engagement_rate"
]

print(df[signals].describe())

       days_since_last_update           ctr  avg_position  search_volume  \
count            30000.000000  30000.000000   30000.00000   27532.000000   
mean                46.098300      0.510733      16.34238     158.882391   
std                 42.078709      3.279162      15.21679    1518.270825   
min                  1.000000      0.000000       0.00000       0.000000   
25%                 20.000000      0.000000       6.20000       0.000000   
50%                 20.000000      0.070000      10.80000      10.000000   
75%                104.000000      0.290000      22.30000      20.000000   
max                373.000000    100.000000     245.00000   74000.000000   

       engagement_rate  
count     30000.000000  
mean          2.534520  
std           8.310096  
min           0.000000  
25%           0.000000  
50%           0.000000  
75%           1.350000  
max         100.000000  


The distributions are uneven, especially for `search_volume` and `engagement_rate`. For example, search volume has a median of 10 but a maximum of 74,000, while engagement rate has a median of 0 and a maximum of 100. This shows that a small number of pages have much larger values than most pages, so simple averages should be interpreted carefully when choosing thresholds for the baseline rule.

## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

### Signal #1: Content staleness

In [ ]:
df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[-1, 30, 90, 180, float("inf")],
    labels=["0-30 days", "31-90 days", "91-180 days", "180+ days"]
)

staleness_test = (
    df.groupby("staleness_bucket", observed=True)["engagement_rate"]
      .agg(["count", "mean"])
      .reset_index()
)

print(staleness_test)

  staleness_bucket  count      mean
0        0-30 days  20480  2.599727
1       31-90 days    175  2.134971
2      91-180 days   9171  2.406133
3        180+ days    174  2.028333


**Question:** Do pages that have gone longer without an update show weaker engagement?

I grouped pages by `days_since_last_update` and compared their average `engagement_rate`.

**Verdict: MIXED**

**Evidence:** Engagement was highest for pages updated within 30 days (mean = 2.600) and lowest for pages older than 180 days (mean = 2.028), suggesting a possible directional relationship. However, the pattern is not consistently monotonic, and the 31–90 day and 180+ day groups contain only 175 and 174 pages respectively. Staleness therefore appears useful as a directional signal, but not as a strong standalone indicator.

### Signal #2: CTR

In [ ]:
df["ctr_bucket"] = pd.qcut(
    df["ctr"],
    q=4,
    duplicates="drop"
)

ctr_test = (
    df.groupby("ctr_bucket", observed=True)["avg_position"]
      .agg(["count", "mean"])
      .reset_index()
)

print(ctr_test)

       ctr_bucket  count       mean
0  (-0.001, 0.07]  15224  19.218431
1    (0.07, 0.29]   7503  15.116007
2   (0.29, 100.0]   7273  11.587323


**Question:** Do pages with lower CTR also tend to have weaker search visibility?

I grouped pages by CTR and compared their average search position.

**Verdict: CONFIRMED**

**Evidence:** The lowest-CTR group had an average position of 19.218, compared with 15.116 for the middle group and 11.587 for the highest-CTR group. Higher CTR is therefore associated with better average search position in this dataset, making CTR a useful directional signal for the baseline ranking.

### Signal #3: Search Volume

In [ ]:
df["volume_bucket"] = pd.qcut(
    df["search_volume"],
    q=4,
    duplicates="drop"
)

volume_test = (
    df.groupby("volume_bucket", observed=True)["impressions_90d"]
      .agg(["count", "mean"])
      .reset_index()
)

print(volume_test)

     volume_bucket  count         mean
0   (-0.001, 10.0]  18392  5547.994182
1     (10.0, 20.0]   2290  5690.318341
2  (20.0, 74000.0]   6850  5796.309635


**Question:** Do pages with higher search volume represent greater opportunity for a refresh?

I grouped pages by search volume and compared their average impressions.

**Verdict: MIXED**

**Evidence:** Average impressions increased from 5,547.99 in the lowest search-volume group to 5,690.32 in the middle group and 5,796.31 in the highest group. The direction is positive, but the differences are relatively small, so search volume appears to provide some opportunity signal without being a strong standalone indicator.

## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

### Flag-linked signal: ***Staleness***

FlyRank's refresh-related flags use staleness as one of their signals. The staleness test showed a mixed relationship between time since update and engagement.

**Verdict: MIXED**

**Evidence:** Pages updated within 30 days had the highest average engagement rate (2.600), while pages older than 180 days had the lowest (2.028). However, the relationship was not consistently monotonic, and the 31–90 day and 180+ day groups were relatively small (175 and 174 pages). Staleness is therefore reasonable to keep as a supporting signal, but the data does not justify treating it as a strong standalone refresh flag.

This is directional evidence only. It does not show that refreshing a page will cause its engagement to improve.

In [ ]:
print("Flag-linked signal: staleness")
print(staleness_test)

Flag-linked signal: staleness
  staleness_bucket  count      mean
0        0-30 days  20480  2.599727
1       31-90 days    175  2.134971
2      91-180 days   9171  2.406133
3        180+ days    174  2.028333


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

The audit suggests that CTR is the strongest of the three tested signals, showing a clear association between higher CTR and better average search position. Staleness and search volume show weaker or mixed relationships, so they should be treated as supporting signals rather than decisive rules. A content team could therefore use a combination of these signals to prioritize pages for review, while treating the resulting ranking as decision-support rather than proof that a page needs refreshing.

## Self-check

Before you submit, confirm each line honestly:

- [✔️] Every section above is filled — markdown thinking AND the code that backs it
- [✔️] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✔️] No client names, URLs, or private queries anywhere
- [✔️] My claims use careful words: observed, measured, directional, decision-support
- [✔️] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.